<div style="text-align: left; margin-bottom: 20px;">
  <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c" 
       alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# DATA/MSML 641: Natural Language Processing
## Session 9: Evaluation II: LLM Benchmarks and LLM-as-a-Judge

**University of Maryland, College Park**  
**Fall 2026**  
**Instructor**: Armin Mehrabian  
**Date**: November 3, 2026

> **Part 2 of 2.** Continues from **Lecture 4: Evaluation I**, now that we have transformers and language models in hand.
> Part 1 covered evaluation foundations and the benchmarking ecosystem; this lecture covers modern LLM benchmarks, learned metrics, contamination, human evaluation, and LLM-as-a-judge.


In [ ]:
# Imports and Setup

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


# Recap: From Part 1

In **Evaluation I** we established:

- **Why, what, and how** we evaluate - formative vs. summative, and data partitioning (train/dev/test, leakage, cross-validation)
- **Classical metrics**: BLEU, ROUGE, precision / recall / F1, and pass@k
- **The benchmarking ecosystem**: shared tasks to benchmarks to leaderboards, Goodhart's law, and how to read a results table critically

Now we turn to **evaluation in the LLM era**, which builds directly on the transformer and language-model machinery from the preceding weeks.


<div style="text-align: left; margin-bottom: 20px;">
 <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c"
 alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# **A. Standard LLM Benchmarks and Metrics**

### How do we measure progress in the state of the art NLP today? 


# Why Benchmarks?

- Benchmarks make NLP evaluation **comparable** and **reproducible** 
    - same dataset + split + metric across all models 

- They let us track **progress over time** and identify **model weaknesses** 

- Typical benchmark families 
 - Knowledge & Reasoning 
 - Math & Logic 
 - Code Generation 
 - Retrieval / Embedding 
 - Text Generation (MT / Summarization / Dialogue)

> Benchmarks are the *institutional memory* of NLP progress.


### Huggingface Open LLM leaderbaord

https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/

# Knowledge & Reasoning Benchmarks

- **MMLU (Massive Multitask Language Understanding)** 
    - 57 subjects (STEM, humanities, law, medicine) 
    - **Format:** 4-choice multiple-choice questions 
    - **Metric:** accuracy (percentage correct) 

- **MMLU-Pro (2024)**: Harder successor with 4–10 options per question 


<div style="text-align:center; margin-top:10px;">
 <img src="img/mmlu_1.png" alt="Example MMLU question" style="max-width:60%; border:1px solid #ccc; margin:5px;">
 <img src="img/mmlu_2.png" alt="MMLU benchmark coverage" style="max-width:60%; border:1px solid #ccc; margin:5px;">
</div>




# SuperGLUE Tasks Explained

Each SuperGLUE subtask targets a **different aspect of language understanding**.
Together, they test reasoning, inference, and coreference: beyond simple pattern matching.

| **Task** | **What It Tests** | **Example Behavior** |
|:--|:--|:--|
| **BoolQ** | Yes/no *reading comprehension* | Read a passage and answer a factual question (e.g., “Is Barq’s root beer a Pepsi product?” → *No*) |
| **CB (CommitmentBank)** | *Textual entailment* with uncertain stance | Judge if a hypothesis follows from a conversational statement (*Entailment / Contradiction / Unknown*) |
| **COPA** | *Causal reasoning* | Choose which event best explains or results from another (“The grass was cut” → The sun was rising ✅) |
| **MultiRC** | *Multi-sentence reasoning* | Answer questions requiring aggregation across multiple sentences |
| **ReCoRD** | *Reading comprehension + commonsense inference* | Fill in missing entities from a passage (“Puerto Rico Gov. Ricardo Rossello said…” → Correct Entity = US) |
| **RTE** | *Recognizing textual entailment* | Determine whether one statement entails another (“Christopher Reeve had an accident” → False) |
| **WiC** | *Word sense disambiguation* | Decide if a target word has the same meaning in two contexts (“board” as noun vs verb → False) |
| **WSC** | *Coreference resolution / commonsense reasoning* | Identify pronoun reference in tricky cases (“He should have been more truthful.” → False) |



>**Takeaway:** 
> SuperGLUE decomposes “language understanding” into eight precise diagnostic tasks: 
> providing a composite view of a model’s linguistic and reasoning competence.


<div style="text-align:center; margin-top:10px;">
 <img src="img/superglue_wang.png" alt="Superglue Wang et al." style="max-width:100%; border:1px solid #ccc; margin:5px;">
</div>

Wang, Alex, et al. "Superglue: A stickier benchmark for general-purpose language understanding systems." 

# Commonsense & Contextual Reasoning – HellaSwag

- **HellaSwag (Zellers et al., ACL 2019)** 
    - Focuses on **commonsense reasoning** and **grounded continuation** 
    - Given a short premise (caption or event), the model must choose the most plausible continuation among 4 candidates 
    - Options are *adversarially filtered*: superficially similar but semantically incorrect 
    - **Metric:** multiple-choice accuracy 

Example:
> *Premise:* “A woman is putting a baby into the crib…” 
> *Options:* 
> A) She closes the crib and turns off the light ✅ 
> B) She lifts the baby and begins a dance performance 
> C) The baby walks out of the house 
> D) The woman paints a wall 

→ Tests whether the model understands everyday physical and social events.


## HellaSwag-Pro (2025): Robustness Evaluation

**HellaSwag-Pro** is a large-scale **bilingual (English + Chinese)** benchmark that extends HellaSwag
to test whether LLMs genuinely *understand* commonsense knowledge or merely exploit memorized surface patterns.

- **11,200 cases** across **7 question variant types**
- Built on top of HellaSwag and a new Chinese HellaSwag (12,000 instances, 56 categories)
- Evaluated **41 representative LLMs**: all showed significant robustness gaps

### The 7 Variant Types

| Variant | What It Tests |
|:--|:--|
| **Problem Restatement** | Rephrase the question; does the model still answer correctly? |
| **Reverse Conversion** | Give the outcome, ask the model to infer the context |
| **Causal Inference** | Identify cause-effect links across events |
| **Sentence Ordering** | Rearrange the event sequence and re-query |
| **Scenario Refinement** | Minimally adjust context to change the correct answer |
| + 2 additional variants | Structural and negation-based perturbations |

### Key Findings
- LLMs **drop substantially** in accuracy across variant types, even when the underlying knowledge is the same
- Performance is **language-dependent**: models robust in English are not always robust in Chinese
- Result: current LLMs are **far from robust** in commonsense reasoning

### Other Real Commonsense Benchmarks
- **WinoGrande** (Sakaguchi et al., 2021): 44k Winograd-schema pronoun resolution items, adversarially filtered 
- **PIQA** (Bisk et al., 2020): physical intuition (binary choice on physical procedures) 
- **ARC-Challenge** (Clark et al., 2018): 3rd–9th grade science questions, adversarially selected

<sub>Liu et al. (2025). "HellaSwag-Pro: A Large-Scale Bilingual Benchmark for Evaluating the Robustness of LLMs in Commonsense Reasoning." ACL Findings 2025. arXiv:2502.11393</sub>


# Math Reasoning Benchmarks

- **GSM8K (Cobbe et al., 2021)** 
    - 8.5 k grade-school math word problems 
    - Each requires **multi-step arithmetic reasoning** 
    - **Input:** natural-language problem 
    - **Output:** final numeric answer 
    - **Metric:** Exact Match (EM): does the predicted final answer exactly match the ground truth? 

- **MATH (Hendrycks et al., 2021)** 
    - Harder dataset from AMC & Olympiad-level problems 
    - Covers algebra → geometry → probability 
    - Evaluated with **Exact Match**, sometimes per-topic breakdown 

### Example (GSM8K)
> *Question:* 
> Janet’s ducks lay 16 eggs per day. She eats 3 for breakfast and uses 4 to bake muffins. She sells the rest for \$2 each. 
> How much does she make per day?

**Reasoning (Chain of Thought):** 
1. Total eggs = 16 
2. Eggs used = 3 + 4 = 7 
3. Eggs sold = 16 − 7 = 9 
4. Revenue = 9 × 2 = \$18 

**Final Answer:** 18 
**Metric:** Exact Match = ✔ (correct)


# Code Generation Benchmarks

- **HumanEval (Chen et al., 2021)** 
    - 164 Python programming problems 
    - **Input:** function signature + docstring 
    - **Output:** valid Python implementation 
    - **Metric:** $pass@k$: probability that at least one of the $k$ generated programs passes all unit tests 

- **MBPP (Austin et al., 2021)** 
    - “Mostly Basic Python Problems” for simpler coding skills 
    - Also uses test-based evaluation rather than string matching 

- **SWE-bench (Jimenez et al., 2024)** 
    - End-to-end software-engineering tasks on real GitHub repositories 
    - Model must edit code to fix a bug or implement a feature 
    - Evaluated by whether all unit tests pass after the patch 



### Example (HumanEval)
> *Prompt:* 
> ```python
> def has_close_elements(numbers: list[float], threshold: float) -> bool:
> """Check if any two numbers are closer than the given threshold."""
> ```
> Model generates several candidate implementations. 
> Each is executed against unit tests, and $pass@k$ is computed.

**Takeaway:** 
Code-generation benchmarks measure *functional correctness*, not surface similarity. 
Metrics like $pass@k$ reflect stochastic diversity: higher $k$ means a greater chance of success.

# Agent Evaluation: tau-bench

**tau-bench** (Yao et al., 2024) evaluates tool-calling agents in realistic multi-turn scenarios.

### Setup
Two domains with real database schemas, APIs, and policies:
| Domain | Users | Products/Flights | Tools | Tasks |
|:--|:--|:--|:--|:--|
| Retail agent | 500 | 50 products, 1k orders | ~10 | 115 |
| Airline agent | 500 | 300 flights, 2k reservations | ~10 | 50 |

### What's Evaluated
The agent must: look up state, apply policy rules, call APIs, and respond correctly  -  across multiple turns with a simulated user.

**Metric:** **pass^k**  -  probability all $k$ independent runs succeed (reliability, not just peak performance).

> A model scoring 80% per run has pass^5 ≈ 0.33. Reliability matters more than single-run accuracy for deployed agents.

<sub>Yao et al. (2024). "τ-bench: A Benchmark for Tool-Agent-User Interaction in Real-World Domains."</sub>

# Retrieval and Embedding Benchmarks

- **BEIR (Thakur et al., 2021)** 
    - Benchmark for zero-shot document retrieval 
    - Includes 18 datasets across diverse domains 
 (news, Wikipedia, QA, biomedical, scientific, web) 
    - Evaluates how well sentence or document embeddings support retrieval 
    - **Metrics:** nDCG@k, Recall@k

- **MTEB (Massive Text Embedding Benchmark, Muennighoff et al., 2023)** 
    - Unified framework for evaluating text embeddings 
    - Covers 8 task categories: 
 STS, classification, clustering, reranking, retrieval, bitext mining, summarization, pair classification 
    - **Metrics:** accuracy, F1, or Spearman $\rho$, depending on task



# Metrics for Retrieval and Embedding Evaluation

### 1. nDCG@k: Normalized Discounted Cumulative Gain
Used in retrieval tasks (e.g., BEIR) to measure *ranking quality*.

$DCG@k = \sum_{i=1}^{k} \dfrac{rel_i}{\log_2(i + 1)}$
 where $rel_i$ is the relevance score of the document at rank $i$.

$ nDCG@k = \dfrac{DCG@k}{IDCG@k} $

**Example**

| Rank | Document | Human Relevance Score | Comment |
|:----:|:---------:|:--------------------:|:--------|
| 1 | Doc A | 3 | Highly relevant |
| 2 | Doc B | 2 | Somewhat relevant |
| 3 | Doc C | 0 | Not relevant |

Ideal ranking (ground truth): [3, 2, 0] 
System ranking (perfect): [3, 2, 0] → $nDCG@3 = 1.0$

If the system ranked [2, 0, 3] instead: 
$DCG@3 = 2/1 + 0/\log_2(3) + 3/\log_2(4) = 3.5$ 
$IDCG@3 = 4.26 \Rightarrow nDCG@3 = 0.82$

→ Higher $nDCG@k$ means relevant items appear earlier in the ranking.

### 2. Spearman ρ: Rank Correlation
Used in embedding evaluation (e.g., MTEB) to measure how well 
model similarity rankings agree with human similarity judgments.

$\rho = 1 - \dfrac{6 \sum d_i^2}{n(n^2 - 1)}$

**Example**

| Pair | Human Rank | Model Rank | $d_i$ | $d_i^2$ |
|:--:|:--:|:--:|:--:|:--:|
| 1 | 1 | 1 | 0 | 0 |
| 2 | 2 | 3 | 1 | 1 |
| 3 | 3 | 2 | –1 | 1 |
| 4 | 4 | 5 | 1 | 1 |
| 5 | 5 | 4 | –1 | 1 |

$\sum d_i^2 = 4$, $n = 5$ 
$\rho = 1 - \dfrac{6 × 4}{5 (25 − 1)} = 0.8$

→ $\rho = 0.8$ means strong agreement between model and human judgments.



**Takeaway** 
- $nDCG@k$ evaluates ranking order for retrieval. 
- $\rho$ measures rank agreement for embeddings. 
Together they capture how well models encode and retrieve meaning.

# Machine Translation Benchmark: WMT

### Overview
- **Workshop on Machine Translation (WMT)**: annual shared task since 2005 
- Evaluates *translation quality* across many language pairs 
- Combines **automatic** (BLEU, COMET) and **human** evaluation 
- Tests both fluency and semantic adequacy

**Example**

Reference: “the cat is on the mat” 
Candidate: “the cat sat on mat” 
→ Overlaps on 4 out of 6 unigrams → BLEU ≈ 0.67

**Interpretation:** 
Higher BLEU → more lexical overlap, but doesn’t capture meaning differences.



**Takeaway** 
BLEU provides quick reproducible comparison, 
but newer metrics (e.g. COMET) better capture semantic quality.

# COMET: Why BLEU Falls Short and How COMET Works


### Why BLEU Falls Short for MT
BLEU treats *any* deviation from the reference as an error: 
it cannot distinguish a fluent but meaning-altering translation from a minor rewording.

### How COMET Works
COMET is a **learned neural metric** trained to predict human quality judgments (MQM scores).

1. **Encoder**: a multilingual pretrained LM (XLM-R) encodes **three** inputs separately:
 - Source sentence $s$
 - Hypothesis (system output) $h$
 - Reference translation $r$
2. **Aggregation**: sentence embeddings are combined (concatenation + element-wise product & difference)
3. **Regression head**: predicts a scalar quality score

**Key advantage over BLEU:** 
COMET uses the **source sentence**, it can detect when a translation is fluent but
diverges from what the source actually says. BLEU has no access to the source.

# COMET Variants and Performance

### COMET Variants (2024)

| Model | Type | Description |
|:--|:--|:--|
| `wmt22-comet-da` | Reference-based | Standard COMET, trained on Direct Assessment scores |
| `wmt22-cometkiwi-da` | Reference-free (QE) | Uses only source + hypothesis: no reference needed |
| **XCOMET-XL** | Reference-based + error spans | Identifies error spans with MQM severity labels |

### Example

| System | BLEU | COMET |
|:--|:--|:--|
| Translation A (fluent but wrong meaning) | 28.4 | 0.42 |
| Translation B (slightly awkward but faithful) | 19.1 | 0.79 |

→ COMET ranks Translation B higher: correctly matching human preference. 
→ BLEU ranks Translation A higher: rewarding surface overlap over semantic adequacy.

**Takeaway:** 
COMET is the current gold standard for MT evaluation at WMT, replacing BLEU as the primary metric. 
It correlates substantially better with human MQM judgments than any n-gram–based metric.

<sub>Rei et al. (2020). "COMET: A Neural Framework for MT Evaluation." EMNLP 2020.</sub>

# From WMT to Summarization: Applying Learned Lessons

The evaluation principles from machine translation transfer directly to **text summarization**:

| MT Concept | Summarization Equivalent |
|:--|:--|
| Reference translation | Human-written summary |
| BLEU (precision) | Used rarely: too strict for abstractive output |
| ROUGE (recall) | **Primary** metric: did the summary cover key content? |
| COMET (learned) | FactCC, QAFactEval, or LLM-judge for factual consistency |

### Why Recall Matters More in Summarization

In translation, you want **precision**: don't add words not in the reference. 
In summarization, you want **recall**: cover the important facts.

A summary that misses the key event is worse than one that adds minor phrasing differences. 
This is why **ROUGE** (recall-oriented) dominates summarization evaluation.



**ROUGE-N** (see earlier section) measures n-gram recall against the reference:

$$ROUGE\text{-}N = \frac{\text{matching n-grams in candidate}}{\text{n-grams in reference}}$$

The **XSum** benchmark below pushes this to its limits with one-sentence summaries.

# XSum: Extreme Summarization

### Overview
- **XSum (BBC Extreme Summarization Corpus)** 
 Each article → **one-sentence summary** capturing the *central event* or *fact*.
- Designed to test:
 - **Abstraction:** generate new phrasing, not copy text
 - **Compression:** condense entire article into a single sentence
 - **Factual grounding:** maintain key information accurately

Evaluated using ROUGE (see earlier section)

# XSum: Limitations and Beyond ROUGE

### Limitations of ROUGE in XSum

- High ROUGE doesn’t guarantee factual accuracy 
 (“EU votes to leave UK” would score similarly!) 
- Penalizes valid paraphrases
- Ignores entity substitutions and hallucinations

### Beyond ROUGE
Modern XSum evaluations include:
- **BERTScore:** contextual embedding similarity 
- **FactCC / QAFactEval:** factual consistency checks 
- **LLM-as-Judge:** GPT-4 rating on faithfulness, coherence, and coverage

**Takeaway** 
XSum tests *extreme abstraction and factual precision*: 
making it a far harder summarization benchmark than CNN/DailyMail or Gigaword.

# BERTScore: How It Works


### Motivation
BLEU penalizes *any* word that doesn't appear in the reference, it treats
"automobile" and "car" as completely different. 
**BERTScore** uses contextual BERT representations, so semantically similar words
score nearly identically.

### How It Works

**Step 1: Embed both sentences** with a pretrained model (e.g., `roberta-large`). 
Each token gets a **contextual vector**: its meaning in that specific sentence.

**Step 2: Greedy matching** via cosine similarity:

$$P_{\text{BERT}} = \frac{1}{|\hat{y}|} \sum_{\hat{y}_i \in \hat{y}} \max_{y_j \in y} \cos(\hat{y}_i,\, y_j)$$

$$R_{\text{BERT}} = \frac{1}{|y|} \sum_{y_j \in y} \max_{\hat{y}_i \in \hat{y}} \cos(\hat{y}_i,\, y_j)$$

$$F_{\text{BERT}} = 2 \cdot \frac{P_{\text{BERT}} \cdot R_{\text{BERT}}}{P_{\text{BERT}} + R_{\text{BERT}}}$$

Where $\hat{y}$ = candidate tokens, $y$ = reference tokens.

**Step 3: IDF weighting (optional):** Rare, informative tokens are upweighted;
common function words ("the", "a") contribute less.

# BERTScore: Example and Comparison

### Example

| Candidate | Reference | BLEU | BERTScore-F1 |
|:--|:--|:--|:--|
| "The car drove fast." | "The automobile sped quickly." | 0.13 | 0.94 |
| "A dog chased the ball." | "The automobile sped quickly." | 0.08 | 0.61 |

→ BERTScore correctly identifies the first paraphrase as high-quality.

### Strengths and Limitations

| | BLEU | BERTScore |
|:--|:--|:--|
| Handles synonyms | ✗ | ✓ |
| Handles paraphrases | ✗ | ✓ |
| Computationally cheap | ✓ | ✗ |
| Interpretable score | ✓ (0–1 precision) | Partial (backbone-dependent) |
| Reference-free option | ✗ | ✗ |

**Takeaway:** 
BERTScore is a stronger automatic metric than BLEU/ROUGE for tasks involving paraphrase
or varied wording: used widely for summarization and dialogue evaluation.

<sub>Zhang et al. (2020). "BERTScore: Evaluating Text Generation with BERT." ICLR 2020. arXiv:1904.09675</sub>

In [11]:
# BERTScore demo
import subprocess, sys

# Install bert_score if not already available
try:
    from bert_score import score as bert_score
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "bert_score", "-q"])
    from bert_score import score as bert_score

candidates = [
    "The automobile drove at high speed.",   # synonym of reference
    "A dog chased the ball.",                 # unrelated
    "The car moved quickly down the road.",   # close paraphrase
]
references = [
    "The car drove at high speed.",
    "The car drove at high speed.",
    "The car drove at high speed.",
]

# Compute BERTScore (using distilbert for speed; roberta-large is standard)
P, R, F1 = bert_score(candidates, references, lang="en",
                      model_type="distilbert-base-uncased", verbose=False)

# Rough unigram precision for comparison
def bleu1(cand, ref):
    c_words = set(cand.lower().split())
    r_words = set(ref.lower().split())
    return len(c_words & r_words) / len(c_words)

print(f"{'Candidate':<45} {'BLEU-1':>8} {'BERTScore F1':>14}")
print("-" * 70)
for cand, ref, f1 in zip(candidates, references, F1.tolist()):
    b = bleu1(cand, ref)
    print(f"{cand:<45} {b:>8.3f} {f1:>14.3f}")

print()
print("→ BERTScore ranks the synonym paraphrase higher than BLEU-1 does.")
print("  It captures semantic similarity that n-gram overlap misses.")


/Users/arminmehrabian/miniconda3/envs/gwu/lib/python3.10/site-packages/huggingface_hub-0.29.2-py3.8.egg/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/Users/arminmehrabian/miniconda3/envs/gwu/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Candidate                                       BLEU-1   BERTScore F1
----------------------------------------------------------------------
The automobile drove at high speed.              0.833          0.979
A dog chased the ball.                           0.200          0.763
The car moved quickly down the road.             0.333          0.851

→ BERTScore ranks the synonym paraphrase higher than BLEU-1 does.
  It captures semantic similarity that n-gram overlap misses.


<div style="text-align:center; margin-top:10px;">
 <img src="img/ai_benchmark_progress.png" alt="AI benchmark progress" style="max-width:80%; border:1px solid #ccc; margin:5px;">
 
</div>

# Benchmark Contamination: A Critical Problem

### What Is Benchmark Contamination?

Benchmark contamination (also called **data contamination** or **train-test overlap**)
occurs when examples from a benchmark's **test set appear in a model's pretraining data**.

Because web-scraped corpora are enormous and unfiltered, many popular benchmarks
have had their questions appear on internet pages, forums, homework-help sites, and GitHub.

**Types of contamination:**
- **Direct**: the exact question-answer pair is in training data
- **Near-duplicate**: paraphrased versions appear in training data
- **Partial**: the answer is in training data even if the question phrasing differs

# Why Contamination Is Systemic

### Why It Is Systemic

1. **Training data opacity**: most frontier models do not publish training data, so
 external contamination auditing is impossible
2. **Incentive misalignment**: labs benefit from high benchmark scores
3. **No consistent standard**: different labs use different overlap-detection thresholds
4. **Benchmark saturation may be artificial**: some apparent "rapid progress" may reflect
 memorization rather than genuine capability improvement

### Evidence

- Masking a wrong answer choice in an MCQ and asking the model to fill it in ("slot guessing"):
 GPT-4 achieves **57% exact match** on MMLU option text: far above the 25% chance baseline
 → indicating memorization of specific answer strings
- Removing contaminated examples from GSM8K evaluation drops model accuracy by up to **13%**
 for some models
- Models trained after AIME 2024 problems circulated online score significantly higher
 than expected, while post-cutoff AIME 2025 scores drop substantially

<sub>Golchin & Surdeanu (2023). "Investigating Data Contamination in Modern Benchmarks." arXiv:2311.09783 
Yang et al. (2024). "Benchmark Data Contamination of Large Language Models: A Survey." arXiv:2406.04244</sub>

# Detecting Contamination


### Detection Methods

| Method | How It Works |
|:--|:--|
| **Slot guessing** | Mask an answer option; if model fills it exactly, it likely memorized it |
| **Min-k% Prob** | High model probability on n-grams → likely in training data (Shi et al. 2024) |
| **Temporal analysis** | Evaluate on problems released *after* model training cutoff |
| **Canary injection** | Embed unique strings in eval sets; check if model can complete them |

# Live Benchmarks and Saturation

### Live / Dynamic Benchmarks: The Solution

The answer to contamination is **continuously updated benchmarks** using newly generated content.

| Benchmark | Design | Why Contamination-Free |
|:--|:--|:--|
| **LiveBench** (2024) | Monthly problems from recent arXiv, news, IMDb, contests | Questions released *after* any training cutoff |
| **LiveCodeBench** (2024) | Programming problems from LeetCode/Codeforces with timestamps | Evaluated on problems released *after* model cutoff |
| **MathArena** (2025) | AIME problems released after model training | Real-time post-cutoff evaluation |

### Benchmark Saturation Timeline

| Benchmark | Year Released | Year ~Saturated |
|:--|:--|:--|
| HellaSwag | 2019 | 2021 |
| SuperGLUE | 2019 | 2022 |
| MMLU | 2020 | 2024 |
| GSM8K | 2021 | 2024 |
| HumanEval | 2021 | 2024–25 |

→ Each benchmark lasts ~2–3 years before top models exceed meaningful discrimination thresholds. 
The community has responded with progressively harder successors: MMLU-Pro, GPQA, Humanity's Last Exam.

**Takeaway:** 
A benchmark score is only meaningful if you know (a) whether training data was audited,
(b) the model's training cutoff vs. the benchmark release date, and (c) whether
live/contamination-free evaluation was performed.

# Frontier Benchmarks: GPQA and HLE

As models saturate existing benchmarks, the community has developed progressively harder evaluations:

### GPQA: Graduate-Level Google-Proof Q&A

- **Rein et al. (2023)** 
- 448 PhD-level multiple-choice questions in biology, chemistry, physics 
- Questions are **intentionally Google-proof**: answers cannot be found by web search 
- Human domain experts score ~65%; non-experts ~34% (random = 25%) 
- GPT-4 (2024): ~39% → frontier models are still well below expert humans

### Humanity's Last Exam (HLE)

- **Center for AI Safety (2025)** 
- 3,000+ questions across 100+ academic disciplines 
- Crowd-sourced from domain experts worldwide 
- Includes proofs, derivations, and open-ended questions: not just multiple choice 
- State-of-the-art models score below 20% on release (2025)

# Frontier Benchmarks: ARC-AGI and Summary

### ARC-AGI (Chollet, 2019 → updated 2024)

- Not text-based: **visual pattern induction from few examples** 
- Tests fluid intelligence: infer rules from 3–5 input-output grid pairs and apply to a new input 
- Specifically designed to be **resistant to memorization and scale** 
- ARC-AGI-2 (2025): frontier models score ~5–15%

| Benchmark | Year | Human Expert | Best Model (2025) | Gap |
|:--|:--|:--|:--|:--|
| MMLU | 2020 | ~90% | 92%+ | **Saturated** |
| GSM8K | 2021 | ~95% | 97%+ | **Saturated** |
| GPQA | 2023 | ~65% | ~55% | 10 pp |
| HLE | 2025 | ~95%+ | ~20% | **Large** |
| ARC-AGI-2 | 2025 | ~95% | ~15% | **Very large** |

**Takeaway:** 
The community is actively building harder benchmarks to stay ahead of model capabilities. 
But each new benchmark faces the same contamination and Goodhart pressures as its predecessors.

<sub>Rein et al. (2023). "GPQA." arXiv:2311.12022. 
Chollet (2019). "On the Measure of Intelligence." arXiv:1911.01547.</sub>

# Model Selection: The Pareto Frontier

In practice, no single model "wins"  -  there are tradeoffs.

**Pareto frontier** = the set of models where you cannot improve on one axis without sacrificing another.

Common tradeoffs:
- Quality vs. cost / latency
- Quality vs. safety
- Quality vs. context length

```
Quality
  │   ●  ●
  │  ●
  │ ●
  │●
  └──────────────── Cost →
     (Pareto frontier: bold dots)
```

**Practical takeaway:** Pick the model on the frontier for your constraint. A model *below* the frontier is strictly dominated  -  pay more and get less.

> Benchmarks give a quality axis. Pricing pages give the cost axis. Chatbot Arena gives organic quality estimates. Use all three.

<div style="text-align: left; margin-bottom: 20px;">
 <img src="https://umd-brand.transforms.svdcdn.com/production/uploads/images/logos-primary.jpg?w=1801&h=601&auto=compress%2Cformat&fit=crop&dm=1613775207&s=71ce45031f9164cb409f11a2e28d8b8c"
 alt="UMD Logo" style="max-width: 300px; height: auto;" />
</div>

# **B. Human Evaluation vs LLM-as-a-Judge**



# Why Human Evaluation?

Automatic metrics (BLEU, ROUGE, COMET) can’t judge:
- Factual correctness 
- Coherence across sentences 
- Stylistic quality or nuance

→ We still rely on **human judgment** as the gold standard.


# Human Evaluation: Common Methods

| Method | What Annotators Do | Used For |
|:--|:--|:--|
| **Likert Scales** | Rate quality on 1–5 or 1–7 | Summarization, dialogue |
| **Pairwise Comparison** | Choose the better output | MT, chatbot responses |
| **Ranking** | Order several outputs | Multi-model studies |

**Example Criteria**
- *Fluency*: grammatical, natural wording 
- *Faithfulness*: preserves meaning 
- *Coherence*: logical flow


# Inter-Annotator Agreement: Validating Human Evaluation

### The Problem: Who Judges the Judges?

Before using human judgments as ground truth, we must ask:
**do different human annotators *agree* on the labels?**

If two annotators disagree 40% of the time on sentiment labels, those labels
are not reliable ground truth, they are noisy approximations.
Poor IAA propagates into model training, benchmark construction, and RLHF preference data.

### Cohen's Kappa: The Standard Measure

Cohen's $\kappa$ measures **agreement beyond chance** between two raters:

$$\kappa = \frac{P_o - P_e}{1 - P_e}$$

Where:
- $P_o$ = **observed** agreement (fraction of items both annotators labeled identically)
- $P_e$ = **expected** agreement by chance (computed from marginal label distributions)

**Interpretation:**

| $\kappa$ | Agreement |
|:--:|:--|
| < 0.00 | Less than chance |
| 0.00 – 0.20 | Slight |
| 0.21 – 0.40 | Fair |
| 0.41 – 0.60 | Moderate |
| 0.61 – 0.80 | **Substantial** ← target for NLP annotation |
| 0.81 – 1.00 | Near-perfect |

# Cohen's Kappa: Worked Example

Two annotators label 100 sentences as Positive / Negative:

| | Annotator B: Positive | Annotator B: Negative | Row Total |
|:--|:--:|:--:|:--:|
| **Annotator A: Positive** | 40 | 10 | 50 |
| **Annotator A: Negative** | 15 | 35 | 50 |
| **Col Total** | 55 | 45 | 100 |

$P_o = (40 + 35)/100 = 0.75$ 
$P_e = (50/100)(55/100) + (50/100)(45/100) = 0.275 + 0.225 = 0.50$ 
$\kappa = (0.75 - 0.50) / (1 - 0.50) = 0.50$ → **Moderate agreement**

# IAA in Practice

- **Toxicity benchmarks** (e.g., Jigsaw): $\kappa$ often < 0.40: toxicity is subjective and culturally variable
- **SQuAD 2.0** (extractive QA): $\kappa$ typically > 0.80: task is well-constrained
- **RLHF preference data**: if annotators agree ~70% of the time on a binary task,
 $\kappa \approx 0.40$ (fair): raising serious questions about reward model reliability

**For multiple raters:** use **Fleiss' Kappa**; for ordinal categories: **Weighted Kappa** or **Krippendorff's Alpha**.

> When evaluating a paper's human evaluation, always ask: *what is the IAA?* 
> If unreported, treat the human evaluation with skepticism.

# What Are We Evaluating? Two Axes

When evaluating an LLM output, separate concerns into two dimensions:

| Axis | Criteria | Example question |
|:--|:--|:--|
| **Task performance** | Usefulness, factuality, relevance | "Did it answer the question correctly?" |
| **Alignment** | Tone, style, safety | "Did it respond appropriately and without harm?" |

A response can score high on one and low on the other:
- Accurate but rude → high performance, low alignment
- Safe and polite but factually wrong → low performance, high alignment

**Why it matters:** Evaluation rubrics must specify which axis they target. BLEU/ROUGE measure task performance only. LLM-as-a-Judge can assess both  -  but needs explicit criteria for each.

# Factuality Evaluation: Fact Decomposition

Automatic metrics like ROUGE can't detect factual errors  -  a fluent, wrong sentence scores just as well as a correct one.

### Approach (Wei et al., 2024)

**Step 1: Decompose** the response into atomic facts.

> "Teddy bears, first created in the 1920s, were named after President Roosevelt after he **proudly** wanted to shoot a captured bear."

Atomic facts:
- Teddy bears were first created in the 1920s. *(weight: 0.3)*
- They were named after Theodore Roosevelt. *(weight: 0.4)*
- Roosevelt was on a hunting trip where a bear was captured. *(weight: 0.2)*
- Roosevelt proudly wanted to shoot the bear. *(weight: 0.1)*

**Step 2: Judge each fact** with a search-augmented LLM.

**Step 3: Compute weighted score**  -  facts 1 and 3 are false → score ≈ 0.50

### Why Weighting?
Not all facts matter equally. Missing the key event ("named after Roosevelt") should penalize more than a minor detail.

<sub>Wei et al. (2024). "Long-form factuality in large language models." arXiv:2403.18802</sub>

# Evaluating Agents: Tool-Use Failure Modes

Standard benchmarks test single-turn outputs. Agents introduce a pipeline that fails in distinct ways.

### The Three Failure Points

```
User prompt → [1. Tool selection] → [2. Tool execution] → [3. Response generation]
```

**1. Tool prediction errors**

| Failure | Example |
|:--|:--|
| Does not use tool | LLM answers from memory when it should call an API |
| Hallucinates tool | Calls `find_bear()` when only `find_teddy_bear()` exists |
| Wrong tool | Calls `send_message()` instead of `find_location()` |
| Wrong argument | Passes `location=(0,0)` instead of the user's actual location |

**2. Tool call errors**  -  the tool itself returns wrong/empty output (fix the tool, not the model)

**3. Response generation errors**  -  model ignores or misreads the tool's result

### Diagnostic Principle
Each failure point has a different remedy: retrain the router, fix the API, upgrade the model, or trim the context. Mixing them up wastes effort.

> tau-bench and SWE-bench measure end-to-end success but don't tell you *where* the pipeline broke. Logging each step is essential for debugging.

# LLM-as-a-Judge

### Idea
Use a strong LLM (e.g., GPT-4, Claude) to act as an evaluator.

### Process
1. Give prompt + outputs + rubric 
2. Ask model to rate or compare 
3. Aggregate scores across samples

### Benefits
- Fast and consistent 
- Correlates highly with humans ($r > 0.8$) 
- Scales to thousands of examples

### Caution
- May prefer its own “style” 
- Verbosity bias → longer = higher score


# Benchmarks Using LLM-as-Judge

### Chatbot Arena
- Real users compare two anonymous chatbots 
- Votes → **Elo ratings** (like chess) 
- Blind, large-scale, preference-based

### MT-Bench
- Structured multi-turn QA and reasoning 
- GPT-4 acts as automatic judge 
- Scores 1–10 on faithfulness, coherence, helpfulness

**Takeaway:** 
Evaluation is moving from fixed metrics → human preference → **LLM-based judgment** at scale.


# LLM-as-a-Judge: Known Biases


### The Problem

LLM judges are fast and scalable, but they carry **systematic biases**
documented across multiple 2024 papers. Using an LLM judge *naively* can
produce misleading evaluation results.

### Documented Biases

| Bias | What Happens | Example |
|:--|:--|:--|
| **Position Bias** | Judge prefers response placed *first* (or last) in a pair, regardless of quality | Swap A and B → ranking reverses for ~25% of pairs |
| **Verbosity Bias** | Longer responses score higher even when length adds no quality | A 500-word answer beats a 150-word answer on the same question |
| **Self-Preference Bias** | GPT-4 prefers GPT-4-style outputs; Claude prefers Claude-style outputs | Evaluating Claude vs GPT-4 with GPT-4 as judge systematically favors GPT-4 |
| **Sycophancy / Authority Bias** | Judge changes score if told "an expert preferred the other answer" | Model capitulates to stated authority rather than maintaining independent judgment |

# LLM-as-a-Judge: Mitigation and Caveats

### Practical Mitigation

| Mitigation | How It Helps |
|:--|:--|
| **Randomize presentation order** and average both orderings | Cancels position bias |
| **Length-controlled scoring** (e.g., AlpacaEval 2 LC) | Fits a regression controlling for length difference; raises Spearman ρ with human votes from 0.94 → 0.98 |
| **Swap-consistency check** | Flag pairs where judge reverses ranking on swap |
| **Multi-judge averaging** | Use 2–3 different LLMs and average to reduce self-preference |
| **Spot-check with humans** | Validate 5–10% of LLM judgments manually to detect systematic errors |

### Chatbot Arena: Crowdsourced Evaluation Caveats

Chatbot Arena (LMSYS) uses real-user blind comparisons, which largely avoids
automated judge biases, but introduces **crowdsourced biases**:
- Convenience sampling: users who voluntarily participate are not representative
- Recency bias: newer models get more traffic, inflating their Elo
- **Vote rigging**: a 2025 paper demonstrated coordinated voting can shift Elo by tens of points

**Takeaway:** 
LLM-as-judge is a valuable tool, but **always report** which judge was used,
whether position was randomized, and whether length-controlled scoring was applied.
Never treat LLM judge scores as equivalent to human evaluation without validation.

<sub>Zheng et al. (2024). "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena." 
Park et al. (2024). "Offsetbias: Capitalizing on Calibrating the LLM-as-Judge." arXiv:2410.21819 
Dubois et al. (2024). "Length-Controlled AlpacaEval." arXiv:2404.04475</sub>

# Statistical Significance: Why It Matters


### The Problem: Is This Improvement Real?

A model scores 85.2% vs. baseline 84.8% on a test set. 
Is this a genuine improvement: or sampling noise?

**Without statistical testing, we cannot know.**

### Why Significance Matters

- Test sets are finite samples; results are random variables 
- Publication bias favors positive results, inflating apparent progress 
- Many NLP "improvements" are < 0.5% absolute on small test sets 
- A difference that is *not* statistically significant should not be reported as an advance

### Common Tests in NLP

| Test | When to Use |
|:--|:--|
| **Paired bootstrap resampling** | Compare two systems on the same test set: most common in NLP |
| **Approximate randomization** | Non-parametric, fast alternative to bootstrap |
| **McNemar's test** | Binary classification: tests if the error patterns differ |
| **Student's t-test** | Only valid if scores are approximately Gaussian (rarely true for NLP) |

# Paired Bootstrap Resampling

### Paired Bootstrap Resampling

**Algorithm:**
1. Compute observed score difference $\delta = \text{score}(A) - \text{score}(B)$
2. Repeat $B = 10{,}000$ times:
 - Sample $n$ examples **with replacement** from the test set
 - Compute $\delta^*_b$ on this bootstrap sample
3. $p$-value $= $ fraction of $\delta^*_b$ where $\delta^*_b \leq 0$ (if testing $A > B$)

**Interpretation:** 
$p < 0.05$ → less than 5% chance the observed advantage is due to chance alone.

> **Rule of thumb:** In NLP papers, aim for $p < 0.05$. 
> Many improvements in the 0.1–0.5% absolute range are **not** statistically significant.

# Opportunities: Help Build and Evaluate AI

Frontier AI labs and companies increasingly rely on **domain experts** to:

- Design challenging evaluation tasks
- Annotate and rate model outputs
- Red-team models for failures and biases
- Build new benchmarks in specialized fields

This is directly connected to everything we covered today: task design, annotation quality, inter-annotator agreement, and what makes a good metric.

### Where to Look

- [**Mercor**](https://t.mercor.com/gkSDo): matches experts with AI evaluation projects at leading labs
- [**Handshake AI Move Program**](https://joinhandshake.com/move-program/referral?referralCode=E39AE8&utm_source=referral): connects students to AI evaluation and annotation roles

> If you have expertise in language, reasoning, or any technical domain, there is real demand for your judgment.

---
<sub>Some content in this lecture draws on structural inspiration from: Amidi & Amidi, "CME 295: Transformers & Large Language Models, Lecture 8: Evaluation," Stanford University, Fall 2025.</sub>

In [12]:
# Paired bootstrap significance test for NLP system comparison
import numpy as np

def paired_bootstrap_test(scores_a, scores_b, n_bootstrap=10_000, seed=42):
    """
    Test whether system A significantly outperforms system B.
    scores_a, scores_b: per-example binary correctness (1=correct, 0=wrong)
    Returns: observed delta, p-value
    """
    rng = np.random.default_rng(seed)
    n = len(scores_a)
    
    # Observed difference
    delta_obs = np.mean(scores_a) - np.mean(scores_b)
    
    # Bootstrap
    delta_boot = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        delta_boot[b] = scores_a[idx].mean() - scores_b[idx].mean()
    
    # p-value: proportion of bootstrap samples where delta <= 0
    p_value = np.mean(delta_boot <= 0)
    return delta_obs, p_value

np.random.seed(42)
n_examples = 500

# System A: 85% accuracy (better)
true_labels   = np.random.randint(0, 2, n_examples)
scores_A = (np.random.rand(n_examples) < 0.85).astype(float)
scores_B = (np.random.rand(n_examples) < 0.80).astype(float)

acc_A = scores_A.mean()
acc_B = scores_B.mean()
delta, p = paired_bootstrap_test(scores_A, scores_B)

print("Paired Bootstrap Significance Test")
print("=" * 50)
print(f"System A accuracy:  {acc_A:.3f}")
print(f"System B accuracy:  {acc_B:.3f}")
print(f"Observed delta:     {delta:+.3f} ({delta*100:+.1f} pp)")
print(f"p-value:            {p:.4f}")
print()
if p < 0.05:
    print(f"Result: System A is SIGNIFICANTLY better than B (p={p:.4f} < 0.05)")
else:
    print(f"Result: Difference is NOT significant (p={p:.4f} >= 0.05)")

# Demonstrate a non-significant case
scores_C = (np.random.rand(n_examples) < 0.851).astype(float)
delta2, p2 = paired_bootstrap_test(scores_A, scores_C)
print()
print(f"System A vs System C (A={scores_A.mean():.3f}, C={scores_C.mean():.3f}):")
print(f"  delta = {delta2:+.4f}, p = {p2:.4f} → {'SIGNIFICANT' if p2 < 0.05 else 'NOT significant'}")
print()
print("→ Even a small difference in accuracy (5 pp) may not be significant on a 500-sample test set.")
print("  Always report confidence intervals or p-values alongside accuracy numbers.")


Paired Bootstrap Significance Test
System A accuracy:  0.844
System B accuracy:  0.786
Observed delta:     +0.058 (+5.8 pp)
p-value:            0.0088

Result: System A is SIGNIFICANTLY better than B (p=0.0088 < 0.05)

System A vs System C (A=0.844, C=0.856):
  delta = -0.0120, p = 0.7151 → NOT significant

→ Even a small difference in accuracy (5 pp) may not be significant on a 500-sample test set.
  Always report confidence intervals or p-values alongside accuracy numbers.
